# 第 1 章：AI 基础与机器学习——完整实验版

> 本 Notebook 是第 1 章实验室的完整参考实现。目标不是罗列 API，而是用可以直接运行、结果清晰的实验建立机器学习直觉。

学习入口：[阅读讲义](../../从AI基础到端侧部署优化/01_AI基础与机器学习.md) · [练习 Notebook](./练习.ipynb) · [验收说明](./README.md) · [真实任务](./真实任务_设备故障预警.md)

你将依次完成：

1. 玩具线性回归：看清 `fit()` 到底学到了什么；
2. 真实数据回归：理解 MAE、RMSE、R² 和泛化；
3. 玩具二分类：第一次读懂混淆矩阵；
4. Iris 三分类：体验标准化、Pipeline 和预测概率；
5. K-Means 聚类：在没有标签的情况下发现群体；
6. Q-learning：通过奖励学会一条最短路径。

建议先运行完整版本，再使用配套的练习版重新实现核心代码。

## 一条贯穿所有实验的主线

监督学习通常遵循同一套流程：

```text
准备数据 → 划分训练/测试集 → 创建模型 → fit → predict → evaluate
```

其中：

- `X`：模型做判断时使用的特征；
- `y`：希望模型预测的目标；
- `fit()`：从训练数据中学习参数；
- `predict()`：使用学到的参数预测新数据；
- 测试集：模拟模型从未见过的新数据。

In [ ]:
# 环境检查
import sys
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import sklearn

print("Python 版本：", sys.version.split()[0])
print("NumPy 版本：", np.__version__)
print("Matplotlib 版本：", matplotlib.__version__)
print("Scikit-learn 版本：", sklearn.__version__)
print("✅ 环境准备完成")

---

# 实验一：最简单的线性回归

先使用一个规律非常清楚的数据集：

```text
X：1, 2, 3, 4, 5, 6
y：2, 4, 6, 8, 10, 12
```

真实规律显然是 `y = 2x`。我们看看模型能否自己学出这个规律。

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# X 必须是二维结构：(样本数, 特征数)
toy_reg_X = np.array([1, 2, 3, 4, 5, 6]).reshape(-1, 1)
toy_reg_y = np.array([2, 4, 6, 8, 10, 12])

toy_X_train, toy_X_test, toy_y_train, toy_y_test = train_test_split(
    toy_reg_X, toy_reg_y, test_size=0.2, random_state=42
)

toy_reg_model = LinearRegression()
toy_reg_model.fit(toy_X_train, toy_y_train)
toy_y_pred = toy_reg_model.predict(toy_X_test)

print("X 的形状：", toy_reg_X.shape)
print("训练集形状：", toy_X_train.shape)
print("测试集形状：", toy_X_test.shape)
print("真实值：", toy_y_test)
print("预测值：", toy_y_pred)
print(f"学到的权重 w：{toy_reg_model.coef_[0]:.3f}")
print(f"学到的偏置 b：{toy_reg_model.intercept_:.3f}")
print(f"MSE：{mean_squared_error(toy_y_test, toy_y_pred):.3f}")
print(f"R²：{r2_score(toy_y_test, toy_y_pred):.3f}")

x_line = np.linspace(0, 7, 100).reshape(-1, 1)
y_line = toy_reg_model.predict(x_line)

plt.figure(figsize=(7, 5))
plt.scatter(toy_X_train[:, 0], toy_y_train, s=90, label="Train")
plt.scatter(toy_X_test[:, 0], toy_y_test, s=120, marker="x", label="Test")
plt.plot(x_line[:, 0], y_line, "r--", label="Learned line")
plt.xlabel("X")
plt.ylabel("y")
plt.title("Toy Linear Regression")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

## 结果怎么理解？

模型形式为：

```text
预测值 = w × X + b
```

本实验中模型应该学到 `w ≈ 2`、`b ≈ 0`。由于数据完全符合直线且没有噪声，MSE 接近 0、R² 接近 1。

> 这只是用来理解流程的理想数据。真实世界的数据通常有噪声、异常值和无法观测的影响因素，因此不会如此完美。

---

# 实验二：真实数据回归

接下来使用 scikit-learn 自带的糖尿病回归数据集。它包含 442 个样本，每个样本有 10 个经过处理的特征，目标是预测一年后的疾病进展指标。

这一次我们不期待完美结果，而是学习如何评价模型在未见数据上的表现。

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.metrics import mean_absolute_error

reg_X, reg_y = load_diabetes(return_X_y=True)

reg_X_train, reg_X_test, reg_y_train, reg_y_test = train_test_split(
    reg_X, reg_y, test_size=0.2, random_state=42
)

print("全部数据：", reg_X.shape)
print("训练集：", reg_X_train.shape)
print("测试集：", reg_X_test.shape)
print("标签形状：", reg_y.shape)
print("✅ 数据准备完成")

## Shape 中的括号是什么意思？

二维特征矩阵的形状写成：

```text
(样本数量, 特征数量)
```

因此 `(442, 10)` 表示 442 个样本、每个样本 10 个特征。`(442,)` 表示标签是一维数组，共有 442 个目标值。

训练集和测试集仍然拥有相同的 10 个特征，只是样本被拆成了约 80% 和 20%。

In [ ]:
# 创建、训练、预测
reg_model = LinearRegression()
reg_model.fit(reg_X_train, reg_y_train)
reg_y_pred = reg_model.predict(reg_X_test)

# 评估
reg_mae = mean_absolute_error(reg_y_test, reg_y_pred)
reg_mse = mean_squared_error(reg_y_test, reg_y_pred)
reg_rmse = np.sqrt(reg_mse)
reg_r2 = r2_score(reg_y_test, reg_y_pred)

print("========== 回归评估结果 ==========")
print(f"MAE  ：{reg_mae:.2f}")
print(f"MSE  ：{reg_mse:.2f}")
print(f"RMSE ：{reg_rmse:.2f}")
print(f"R²   ：{reg_r2:.3f}")

print("\n前 10 个测试样本：")
print("序号\t真实值\t预测值\t绝对误差")
for i in range(10):
    error = abs(reg_y_test[i] - reg_y_pred[i])
    print(f"{i + 1}\t{reg_y_test[i]:.1f}\t{reg_y_pred[i]:.1f}\t{error:.1f}")

In [ ]:
# 用两个图观察预测质量
reg_residuals = reg_y_test - reg_y_pred
reg_low = min(reg_y_test.min(), reg_y_pred.min())
reg_high = max(reg_y_test.max(), reg_y_pred.max())

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(reg_y_test, reg_y_pred, alpha=0.75, edgecolors="black")
axes[0].plot([reg_low, reg_high], [reg_low, reg_high], "r--", label="Perfect prediction")
axes[0].set_xlabel("Actual Value")
axes[0].set_ylabel("Predicted Value")
axes[0].set_title("Actual vs Predicted")
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].scatter(reg_y_pred, reg_residuals, alpha=0.75, edgecolors="black")
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_xlabel("Predicted Value")
axes[1].set_ylabel("Residual: Actual - Predicted")
axes[1].set_title("Residual Plot")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 四个回归指标分别回答什么？

| 指标 | 它回答的问题 | 判断方向 |
|---|---|---|
| MAE | 平均相差多少？ | 越小越好 |
| MSE | 对大误差加重惩罚后，平均误差多大？ | 越小越好 |
| RMSE | 典型误差大约是多少，且单位与目标一致？ | 越小越好 |
| R² | 模型解释了多少数据变化？ | 越接近 1 越好 |

预测散点越接近红色虚线越好；残差应该尽量围绕 0 随机分布。如果残差呈明显曲线，往往说明线性模型没有捕捉到某种非线性规律。

---

# 实验三：最简单的二分类

现在把任务改成预测类别。输入为 1 到 20，前 10 个标签为 0，后 10 个标签为 1。逻辑回归会学习两类之间的分界位置。

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

toy_clf_X = np.arange(1, 21).reshape(-1, 1)
toy_clf_y = np.array([0] * 10 + [1] * 10)

toy_clf_X_train, toy_clf_X_test, toy_clf_y_train, toy_clf_y_test = train_test_split(
    toy_clf_X, toy_clf_y, test_size=0.2, random_state=42, stratify=toy_clf_y
)

toy_clf_model = LogisticRegression(max_iter=1000)
toy_clf_model.fit(toy_clf_X_train, toy_clf_y_train)
toy_clf_y_pred = toy_clf_model.predict(toy_clf_X_test)
toy_clf_probability = toy_clf_model.predict_proba(toy_clf_X_test)[:, 1]

print("真实类别：", toy_clf_y_test)
print("预测类别：", toy_clf_y_pred)
print("属于类别 1 的概率：", np.round(toy_clf_probability, 3))
print("混淆矩阵：")
print(confusion_matrix(toy_clf_y_test, toy_clf_y_pred))
print("分类报告：")
print(classification_report(toy_clf_y_test, toy_clf_y_pred, zero_division=0))

x_probability = np.linspace(1, 20, 200).reshape(-1, 1)
p_class_1 = toy_clf_model.predict_proba(x_probability)[:, 1]

plt.figure(figsize=(8, 5))
plt.scatter(toy_clf_X[:, 0], toy_clf_y, s=70, label="Samples")
plt.plot(x_probability[:, 0], p_class_1, color="red", label="P(class=1)")
plt.axhline(0.5, color="gray", linestyle="--", label="Threshold=0.5")
plt.xlabel("X")
plt.ylabel("Probability / Class")
plt.title("Toy Logistic Regression")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

## 混淆矩阵再次复习

scikit-learn 对标签 `0、1` 的默认排列是：

```text
[[TN, FP],
 [FN, TP]]
```

- 对角线 `TN、TP`：预测正确；
- 非对角线 `FP、FN`：预测错误；
- FP：误报；
- FN：漏报。

这个玩具数据非常容易分开，因此结果可能全部正确。真实分类任务会更有挑战。

---

# 实验四：Iris 鸢尾花三分类

Iris 是机器学习最经典的入门分类数据集：

- 150 个样本；
- 4 个数值特征；
- 3 种鸢尾花类别。

这里使用 Pipeline 把标准化和逻辑回归串成一个完整流程。这样预处理只会在训练数据上学习参数，可以降低数据泄漏风险。

In [ ]:
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import ConfusionMatrixDisplay

iris = load_iris()
iris_X = iris.data
iris_y = iris.target

iris_X_train, iris_X_test, iris_y_train, iris_y_test = train_test_split(
    iris_X, iris_y, test_size=0.2, random_state=42, stratify=iris_y
)

iris_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000)
)

iris_model.fit(iris_X_train, iris_y_train)
iris_y_pred = iris_model.predict(iris_X_test)
iris_probabilities = iris_model.predict_proba(iris_X_test)

print("数据形状：", iris_X.shape)
print("特征名称：", iris.feature_names)
print("类别名称：", iris.target_names)
print("训练集类别数量：", np.bincount(iris_y_train))
print("测试集类别数量：", np.bincount(iris_y_test))

In [ ]:
iris_accuracy = accuracy_score(iris_y_test, iris_y_pred)
iris_cm = confusion_matrix(iris_y_test, iris_y_pred)

print("========== Iris 分类评估 ==========")
print(f"Accuracy：{iris_accuracy:.3f}")
print("混淆矩阵：")
print(iris_cm)
print("\n分类报告：")
print(classification_report(
    iris_y_test,
    iris_y_pred,
    target_names=iris.target_names,
    zero_division=0
))

ConfusionMatrixDisplay(
    confusion_matrix=iris_cm,
    display_labels=iris.target_names
).plot(cmap="Blues", values_format="d")
plt.title("Iris Confusion Matrix")
plt.show()

In [ ]:
# 查看前 8 个样本的真实类别、预测类别和最大预测概率
print("序号\t真实类别\t预测类别\t置信度\t结果")
for i in range(8):
    actual_name = iris.target_names[iris_y_test[i]]
    predicted_name = iris.target_names[iris_y_pred[i]]
    confidence = iris_probabilities[i].max()
    result = "正确" if iris_y_test[i] == iris_y_pred[i] else "错误"
    print(f"{i + 1}\t{actual_name:10s}\t{predicted_name:10s}\t{confidence:.3f}\t{result}")

print("\n第一个测试样本属于三个类别的概率：")
for class_name, probability in zip(iris.target_names, iris_probabilities[0]):
    print(f"{class_name:10s}: {probability:.3f}")
print("概率之和：", iris_probabilities[0].sum())

## 从预测概率到最终类别

多分类逻辑回归会为每个类别输出一个概率，三个概率之和为 1。`predict()` 最终选择概率最大的类别。

需要注意：

- 高概率不等于绝对正确；
- Accuracy 只反映整体正确比例；
- 混淆矩阵能告诉我们具体是哪些类别发生了混淆；
- Precision、Recall 和 F1 可以从不同角度衡量每个类别。

---

# 实验五：K-Means 聚类

聚类属于无监督学习：训练数据只有 `X`，没有人工标签 `y`。K-Means 会反复执行两步：

1. 把每个样本分给最近的聚类中心；
2. 根据当前分组重新计算聚类中心。

直到中心基本不再变化。

In [ ]:
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

cluster_X, _hidden_labels = make_blobs(
    n_samples=300, centers=3, cluster_std=0.85, random_state=42
)

print("数据形状：", cluster_X.shape)
print("K-Means 训练时不会使用 _hidden_labels")

plt.figure(figsize=(7, 5))
plt.scatter(cluster_X[:, 0], cluster_X[:, 1], color="gray", alpha=0.7, edgecolors="black")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Unlabeled Data Before Clustering")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
cluster_model = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels = cluster_model.fit_predict(cluster_X)
cluster_centers = cluster_model.cluster_centers_
cluster_score = silhouette_score(cluster_X, cluster_labels)

print("========== K-Means 聚类结果 ==========")
print(f"轮廓系数：{cluster_score:.3f}")
print("各簇样本数量：", np.bincount(cluster_labels))
print("聚类中心：")
for index, center in enumerate(cluster_centers):
    print(f"Cluster {index}: ({center[0]:.2f}, {center[1]:.2f})")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(cluster_X[:, 0], cluster_X[:, 1], color="gray", alpha=0.7, edgecolors="black")
axes[0].set_title("Before Clustering")
axes[0].set_xlabel("Feature 1")
axes[0].set_ylabel("Feature 2")
axes[0].grid(alpha=0.3)

axes[1].scatter(
    cluster_X[:, 0], cluster_X[:, 1],
    c=cluster_labels, cmap="viridis", alpha=0.75, edgecolors="black"
)
axes[1].scatter(
    cluster_centers[:, 0], cluster_centers[:, 1],
    marker="*", s=400, color="red", edgecolors="black", label="Centers"
)
axes[1].set_title("After K-Means")
axes[1].set_xlabel("Feature 1")
axes[1].set_ylabel("Feature 2")
axes[1].grid(alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# 尝试不同的 K，使用轮廓系数辅助选择
candidate_k = range(2, 7)
silhouette_scores = []

for k in candidate_k:
    candidate_model = KMeans(n_clusters=k, random_state=42, n_init=10)
    candidate_labels = candidate_model.fit_predict(cluster_X)
    score = silhouette_score(cluster_X, candidate_labels)
    silhouette_scores.append(score)
    print(f"K={k}，轮廓系数={score:.3f}")

best_k = list(candidate_k)[int(np.argmax(silhouette_scores))]
print(f"\n当前候选范围内，轮廓系数最高的 K：{best_k}")

plt.figure(figsize=(7, 4))
plt.plot(list(candidate_k), silhouette_scores, marker="o", linewidth=2)
plt.xlabel("Number of Clusters K")
plt.ylabel("Silhouette Score")
plt.title("Choosing K with Silhouette Score")
plt.xticks(list(candidate_k))
plt.grid(alpha=0.3)
plt.show()

## 聚类结果不能机械地当作真实类别

K-Means 输出的 `0、1、2` 只是簇编号：

- 编号大小没有顺序意义；
- 再训练一次时编号可能互换；
- 聚类发现的是相似性结构，不保证等同于人类定义的类别；
- `n_clusters` 是超参数，需要结合指标、可视化和业务知识选择。

轮廓系数接近 1 表示簇内较紧密、簇间较分离；接近 0 表示不同簇可能重叠。

---

# 实验六：Q-learning 一维寻宝

强化学习不给智能体标准答案，而是让它通过与环境互动获得奖励：

```text
起点                         宝藏
 0  →  1  →  2  →  3  →  4  →  5
```

- 状态：当前位置；
- 动作：向左或向右；
- 普通移动：奖励 -0.1；
- 到达宝藏：奖励 +10；
- 目标：学会尽快到达状态 5。

## Q 表和更新规则

Q 表的每一行对应一个状态，每一列对应一个动作。数值越大，代表智能体认为在该状态执行该动作越值得。

核心更新可以写成：

```text
新 Q 值 = 旧 Q 值 + 学习率 × (学习目标 - 旧 Q 值)
```

其中学习目标包含当前奖励和未来状态可能获得的奖励。

In [ ]:
# 固定随机数，保证结果可以复现
rl_rng = np.random.default_rng(42)

N_STATES = 6
GOAL_STATE = 5
LEFT = 0
RIGHT = 1
ACTION_NAMES = ["LEFT", "RIGHT"]

learning_rate = 0.1
gamma = 0.95
epsilon = 1.0
epsilon_min = 0.05
epsilon_decay = 0.99
episodes = 500
max_steps = 30

q_table = np.zeros((N_STATES, 2))
episode_rewards = []
success_history = []

for episode in range(episodes):
    state = 0
    total_reward = 0.0
    success = 0

    for step in range(max_steps):
        # epsilon-greedy：探索新动作，或利用当前最佳经验
        if rl_rng.random() < epsilon:
            action = int(rl_rng.integers(2))
        else:
            action = int(np.argmax(q_table[state]))

        if action == LEFT:
            next_state = max(0, state - 1)
        else:
            next_state = min(GOAL_STATE, state + 1)

        done = next_state == GOAL_STATE
        reward = 10.0 if done else -0.1

        if done:
            target = reward
        else:
            target = reward + gamma * np.max(q_table[next_state])

        # Q-learning 的核心更新
        q_table[state, action] += learning_rate * (
            target - q_table[state, action]
        )

        state = next_state
        total_reward += reward

        if done:
            success = 1
            break

    episode_rewards.append(total_reward)
    success_history.append(success)
    epsilon = max(epsilon_min, epsilon * epsilon_decay)

print("✅ Q-learning 训练完成")

In [ ]:
print("========== 学到的 Q 表 ==========")
print("状态\t向左Q值\t向右Q值\t最佳动作")
for state in range(GOAL_STATE):
    best_action = int(np.argmax(q_table[state]))
    print(
        f"{state}\t{q_table[state, LEFT]:.3f}\t\t"
        f"{q_table[state, RIGHT]:.3f}\t\t{ACTION_NAMES[best_action]}"
    )

# 关闭探索，只使用学到的最佳策略
state = 0
learned_path = [state]

for _ in range(20):
    action = int(np.argmax(q_table[state]))
    if action == LEFT:
        state = max(0, state - 1)
    else:
        state = min(GOAL_STATE, state + 1)
    learned_path.append(state)
    if state == GOAL_STATE:
        break

print("\n智能体学到的路径：")
print(" → ".join(map(str, learned_path)))
print("✅ 成功到达宝藏" if learned_path[-1] == GOAL_STATE else "❌ 未到达宝藏")

In [ ]:
# 平滑曲线，观察学习过程
window = 25
moving_rewards = np.convolve(
    episode_rewards, np.ones(window) / window, mode="valid"
)
moving_success = np.convolve(
    success_history, np.ones(window) / window, mode="valid"
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(moving_rewards, color="blue")
axes[0].set_title("Moving Average Reward")
axes[0].set_xlabel("Episode")
axes[0].set_ylabel("Average Reward")
axes[0].grid(alpha=0.3)

axes[1].plot(moving_success, color="green")
axes[1].set_title("Success Rate")
axes[1].set_xlabel("Episode")
axes[1].set_ylabel("Success Rate")
axes[1].set_ylim(-0.05, 1.05)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 强化学习参数的直觉

| 参数 | 含义 | 过小时可能怎样 | 过大时可能怎样 |
|---|---|---|---|
| learning_rate | 接受新经验的速度 | 学得慢 | Q 值可能波动 |
| gamma | 对未来奖励的重视程度 | 只顾眼前 | 非常重视长期回报 |
| epsilon | 随机探索的概率 | 容易困在已有策略 | 随机行动太多 |
| step reward | 每走一步的代价 | 不在乎绕路 | 强烈追求最短路径 |

训练前期 epsilon 较大，智能体更多地探索；训练后期 epsilon 逐渐减小，更多地使用已经学到的最佳动作。这就是探索与利用之间的平衡。

---

# 六个实验放在一起看

| 实验 | 学习类型 | 是否有标签 | 输出 | 关键评价方法 |
|---|---|---|---|---|
| 玩具线性回归 | 监督学习 | 有 | 连续数值 | MSE、R² |
| 糖尿病回归 | 监督学习 | 有 | 连续数值 | MAE、RMSE、R² |
| 玩具二分类 | 监督学习 | 有 | 0 / 1 | 混淆矩阵、F1 |
| Iris 三分类 | 监督学习 | 有 | 三种类别 | Accuracy、混淆矩阵 |
| K-Means | 无监督学习 | 没有 | 簇编号 | 轮廓系数、可视化 |
| Q-learning | 强化学习 | 奖励代替标签 | 动作策略 | 累计奖励、成功率 |

真正需要记住的不是每个 API，而是：

1. 先判断任务类型；
2. 明确特征、标签、状态、动作或奖励；
3. 训练和测试必须分开；
4. 根据任务选择合适的指标；
5. 不只看一个数字，还要观察模型具体错在哪里。

完成本 Notebook 后，请关闭参考实现，打开 [`练习.ipynb`](./练习.ipynb)，不看这里的代码重新完成四个核心实验。

---

# 参考实现统一验收

下面的单元会调用与练习版相同的验收器。这里的目的不是给学习者发成绩，而是证明参考实现与当前验收规则保持一致。

In [ ]:
from pathlib import Path
import sys

lab_candidates = [
    Path.cwd(),
    Path.cwd() / "labs" / "01_ai_basics_ml",
    Path.cwd().parent / "labs" / "01_ai_basics_ml",
]
lab_dir = next((path for path in lab_candidates if (path / "validate.py").exists()), None)
if lab_dir is None:
    raise FileNotFoundError("找不到 validate.py，请从仓库根目录或本实验目录启动 Jupyter。")
if str(lab_dir) not in sys.path:
    sys.path.insert(0, str(lab_dir))

from validate import print_report, save_report, validate_exercise

reference_report = validate_exercise(globals())
print_report(reference_report)
if reference_report["passed"]:
    result_path = save_report(reference_report, lab_dir / "artifacts" / "reference_result.json")
    print("参考实现证据已保存：", result_path)

## 下一步

1. 关闭本参考实现；
2. 独立完成 [`练习.ipynb`](./练习.ipynb)；
3. 通过统一验收后，完成[端侧设备故障预警真实任务](./真实任务_设备故障预警.md)。